# Pipeline de dados — trajetórias AIS

Notebook de apoio ao `src/main.py`: explica cada passo e chama as funções dos
módulos em vez de repetir código.

**Como usar:** abre a partir da raiz do repositório
(`jupyter notebook src/pipeline.ipynb`). A célula seguinte encontra a raiz e
`src/` automaticamente, com *working directory* na raiz ou em `src/`.

> Os passos 1, 2 e 4 reescrevem ficheiros em `data/`; são reprodutíveis a
> partir do CSV original.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "data").is_dir() and (ROOT.parent / "data").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

DATA = ROOT / "data"
print("raiz:", ROOT)
print("dados:", DATA)


raiz: /home/dinisrosa22/Documentos/UNI/UC/MIACD/IA CENTRADA NO HUMANO/TRABALHO FINAL 2
dados: /home/dinisrosa22/Documentos/UNI/UC/MIACD/IA CENTRADA NO HUMANO/TRABALHO FINAL 2/data


## 1. Limpeza do CSV

`limpar_csv_para_parcelar` lê só as colunas úteis, ordena por `(id, t)`,
converte o rótulo para binário (`fishing` → 1) e grava `ais_123.parquet`.


In [2]:
from limpar_csv import limpar_csv_para_parcelar

ais_path = limpar_csv_para_parcelar(
    DATA / "123_fishing_trajs.csv",
    DATA / "ais_123.parquet",
)
ais_path


'/home/dinisrosa22/Documentos/UNI/UC/MIACD/IA CENTRADA NO HUMANO/TRABALHO FINAL 2/data/ais_123.parquet'

## 2. Divisão, `t_rel` e normalização

A divisão é feita por **trajetória completa** (70/15/15, semente 42), para não
haver fuga de informação. O `t_rel` fica em bruto e a normalização (mediana e
IQR) é ajustada **só ao treino**; os parâmetros ficam em `normalizacao.json`.


In [3]:
from preparar_dados import preparar_dados

treino_path, teste_path, validacao_path, params_path = preparar_dados(
    DATA / "ais_123.parquet",
    DATA / "treino_norm.parquet",
    DATA / "teste_norm.parquet",
    DATA / "validacao_norm.parquet",
    70, 15, 15,
    params_path=DATA / "normalizacao.json",
)
treino_path, teste_path, validacao_path, params_path


('/home/dinisrosa22/Documentos/UNI/UC/MIACD/IA CENTRADA NO HUMANO/TRABALHO FINAL 2/data/treino_norm.parquet',
 '/home/dinisrosa22/Documentos/UNI/UC/MIACD/IA CENTRADA NO HUMANO/TRABALHO FINAL 2/data/teste_norm.parquet',
 '/home/dinisrosa22/Documentos/UNI/UC/MIACD/IA CENTRADA NO HUMANO/TRABALHO FINAL 2/data/validacao_norm.parquet',
 '/home/dinisrosa22/Documentos/UNI/UC/MIACD/IA CENTRADA NO HUMANO/TRABALHO FINAL 2/data/normalizacao.json')

### Verificação da divisão

`verificar_divisao` confirma que nenhuma trajetória foi partida e que os pontos
de pesca se mantêm.


In [4]:
from verificar_divisao import verificar_divisao

verificacao = verificar_divisao(
    DATA / "ais_123.parquet",
    treino_path, teste_path, validacao_path,
)
verificacao["ok"]


OK    nenhum id aparece em mais de um conjunto
OK    ids do original = ids dos conjuntos (123 trajetórias)
OK    total de pontos: 1026279 (original: 1026279)
OK    todas as trajetórias completas
OK    pontos de pesca preservados (775521 / 775521)

treino       86 trajetórias |   706522 pontos | pesca: 75.6%
teste        18 trajetórias |   146225 pontos | pesca: 73.5%
validacao    19 trajetórias |   173532 pontos | pesca: 77.1%

RESULTADO: tudo OK


True

### Distribuição das features normalizadas

Mediana e IQR foram ajustados só ao treino; as estatísticas dos três conjuntos
devem ficar próximas de 0 e numa escala comparável.


In [5]:
import pandas as pd

treino = pd.read_parquet(treino_path)
features = ["x", "y", "signed_turn", "time_gap", "euc_speed", "distanceToShore"]
treino[features].describe(percentiles=[.01, .05, .5, .95, .99]).T[
    ["min", "1%", "50%", "99%", "max"]
]


,min,1%,50%,99%,max
x,-0.953654,-0.832138,0.0,0.796862,0.810068
y,-1.737895,-1.293382,0.0,0.571052,0.591665
signed_turn,-33.792870,-22.729733,0.0,23.055597,33.792866
time_gap,-1.500000,-1.500000,0.0,5.166667,7860.333496
euc_speed,-1.142955,-1.113662,0.0,2.966880,19.218838
distanceToShore,-0.647019,-0.646422,0.0,1.549458,2.207201


### Porque cortar em gaps > 10 min?

Gaps longos podem ser falhas de transmissão. Se uma janela os atravessasse,
juntaria pontos que não são contíguos no tempo. O maior gap do dataset é uma
falha de transmissão (13,1 h com apenas 1,55 km), não duas viagens fundidas.


In [6]:
from analisar_gaps import analisar_gaps, resumo_gaps

gaps = analisar_gaps(ais_path)
resumo_gaps(gaps).head(5)


,n_gaps,gap_max_h,gap_total_h,dist_max_km
id,,,,
219012671-1,1,13.103333,13.103333,1.555256
232020276-2,30,4.258889,20.695833,19.257429
232020276-3,45,3.982500,29.763334,9.888118
220099000-3,2,2.241945,3.255556,1.764886
220307000-1,1,1.969167,1.969167,9.562455


## 3. Janelamento com sobreposição

Cada trajetória é cortada em janelas de **30 min** com **50% de sobreposição**.
Parâmetros decididos:

- `gap_max_s=600`: nenhuma janela atravessa gaps > 10 min;
- `min_pontos=30`: evita janelas quase vazias;
- `T` não é fixo: o maior nº de pontos define `T` (590 no treino) e as restantes
  janelas levam padding (`mascara=False`, `y=-1`).

O modo alternativo `n_pontos=256` fica para comparação.


In [7]:
from criar_janelas import criar_janelas

r = criar_janelas(
    treino_path,
    duracao_s=30 * 60,
    sobreposicao_pct=50,
    gap_max_s=600,
    min_pontos=30,
)
r["info"]


{'modo': 'tempo',
 'n_pontos': None,
 'duracao_s': 1800,
 'sobreposicao_pct': 50,
 'passo': 900.0,
 'gap_max_s': 600,
 'n_trajetorias': 86,
 'n_janelas': 7685,
 'T': 590,
 'pontos_por_janela_min_mediana_max': (30, 174.0, 590),
 'pontos_descartados': 2617,
 'trajetorias_sem_janelas': []}

### O que observar

- `X`: `(nº de janelas, T, nº de features)`.
- `mascara`: `True` nos pontos reais; o resto é padding.
- `meta`: `id`, posições e `t_rel` de cada janela (para reconstruir segmentos).


In [8]:
n_pontos_treino = len(treino)
sem = r["info"]["pontos_descartados"]
print("X:", r["X"].shape)
print(f"pontos sem janela: {sem:,} de {n_pontos_treino:,} "
      f"({100 * sem / n_pontos_treino:.2f}%)")
print(f"padding: {100 * (1 - r['mascara'].mean()):.1f}% de X")
r["meta"]["n_pontos"].describe(percentiles=[.05, .5, .95])


X: (7685, 590, 6)
pontos sem janela: 2,617 de 706,522 (0.37%)
padding: 69.2% de X


count    7685.000000
mean      181.806376
std        90.705549
min        30.000000
5%         52.000000
50%       174.000000
95%       364.000000
max       590.000000
Name: n_pontos, dtype: float64

### Comparação com o modo pontos

Mesma sobreposição e mesmo corte em gaps, mas janelas de 256 pontos:


In [9]:
r_pontos = criar_janelas(
    treino_path, n_pontos=256, sobreposicao_pct=50, gap_max_s=600,
)
r_pontos["info"]


{'modo': 'pontos',
 'n_pontos': 256,
 'duracao_s': None,
 'sobreposicao_pct': 50,
 'passo': 128,
 'gap_max_s': 600,
 'n_trajetorias': 86,
 'n_janelas': 5436,
 'T': 256,
 'pontos_por_janela_min_mediana_max': (256, 256.0, 256),
 'pontos_descartados': 3376,
 'trajetorias_sem_janelas': []}

## 4. Gerar e guardar as janelas finais

`gerar_janelas` aplica os mesmos parâmetros aos três conjuntos, grava os
`.npz` comprimidos (`janelas_treino.npz`, `janelas_validacao.npz`,
`janelas_teste.npz`), confere a integridade de cada ficheiro e escreve o
resumo em `janelas_resumo.json` (janelas, `T`, cobertura e padding).

Os caminhos são passados a partir de `DATA` para funcionar com qualquer
*working directory*.


In [10]:
import pandas as pd

from gerar_janelas import gerar_janelas

conjuntos = {
    "treino": (str(DATA / "treino_norm.parquet"), str(DATA / "janelas_treino.npz")),
    "validacao": (str(DATA / "validacao_norm.parquet"), str(DATA / "janelas_validacao.npz")),
    "teste": (str(DATA / "teste_norm.parquet"), str(DATA / "janelas_teste.npz")),
}

resumo = gerar_janelas(
    conjuntos=conjuntos,
    resumo_path=str(DATA / "janelas_resumo.json"),
)
pd.DataFrame(resumo["conjuntos"]).T[
    ["n_janelas", "T", "pontos_descartados", "cobertura", "padding", "problemas"]
]

,n_janelas,T,pontos_descartados,cobertura,padding,problemas
treino,7685,590,2617,0.996296,0.691854,[]
validacao,1949,531,79,0.999545,0.666305,[]
teste,1619,481,625,0.995726,0.630125,[]


## 5. Dataset e DataLoader

`DatasetJanelas` lê as janelas do `.npz` e devolve, por item, `X`, `y`,
`mascara`, `length` e `idx`. O `time_gap` do 1.º ponto é zerado **em memória**
apenas nas janelas que começam depois de um gap > 600 s; o `.npz` fica intacto.

`criar_dataloader` agrupa janelas de tamanho parecido (*bucketing*) e usa um
`collate` que corta cada batch ao maior comprimento real, pronto para
`pack_padded_sequence`.

A célula seguinte corre o teste do módulo: formas, máscara, `time_gap[0]`,
loss mascarada e packing.

In [11]:
from testar_dataset_janelas import main as testar_dataset_janelas

testar_dataset_janelas()

7685 janelas, T=590, features=['x', 'y', 'signed_turn', 'time_gap', 'euc_speed', 'distanceToShore']
[ok] formas; 120/121 batches cortados abaixo de T
[ok] padding por batch: 3.9% com bucketing vs 62.5% sem
[ok] máscara coerente com lengths e y == -1
[ok] time_gap[0]=0 em 34 janelas pós-gap; |time_gap[0]| máx. agora = 148.5 (antes: 7860.3)
[ok] loss mascarada = 0.8070 (== BCE só nos pontos reais)
[ok] pack_padded_sequence + BiGRU: saída (64, 162, 16)

Tudo OK.


## 6. Investigação da qualidade dos dados

`investigar_qualidade` verifica nulos, timestamps duplicados, desalinhamentos
`t`/`time_gap`, outliers de velocidade e prefixos compartilhados entre trajetórias
e conjuntos. Esta análise usa apenas os dados locais e não seleciona parâmetros.
O resumo é guardado em `data/qualidade_dados.json`.

In [ ]:
from investigar_qualidade import investigar_qualidade

investigacao = investigar_qualidade()
investigacao

## 7. Baselines ponto a ponto

Os baselines são ajustados **exclusivamente no conjunto de treino**. A célula
não abre, avalia ou usa os conjuntos de validação e teste.

In [ ]:
from baselines_ponto import executar_baselines

baselines_treino = executar_baselines()
baselines_treino

## Próximos passos

- decidir a política para os pontos sem janela (3.4);
- passar ao classificador (secção 5) e às ablações de sobreposição, duração e
  features (3.6).

Plano completo em
[`docs/markdown/passos_futuros.md`](../docs/markdown/passos_futuros.md).